In [4]:
import requests
import json
import os
from dotenv import load_dotenv

# api key 로딩
load_dotenv()

# 환경 변수에서 API 키 로드
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
API_URL = "https://api.upstage.ai/v1/chat/completions"


# --- LangChain 'invoke' 스타일과 유사하게 수정된 함수 ---

def solar_invoke(prompt: str) -> str:
    """
    Upstage SOLAR 모델에 요청을 보내고 최종 응답 문자열을 반환합니다.
    (LangChain llm.invoke()와 유사한 사용 패턴)
    """
    if not UPSTAGE_API_KEY:
        print("Error: UPSTAGE_API_KEY not found in environment variables.")
        return "API Key Error"

    headers = {
        'Authorization': f'Bearer {UPSTAGE_API_KEY}',
        'Content-Type': 'application/json',
    }

    data = {
        "model": "solar-pro2",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        # 'stream': False (스트리밍을 비활성화하고 최종 응답을 한 번에 받습니다)
    }

    try:
        # stream=False가 기본이므로, 한 번에 전체 응답을 받습니다.
        response = requests.post(
            API_URL,
            headers=headers,
            data=json.dumps(data)
        )

        # HTTP 에러 처리 (4xx 또는 5xx 상태 코드)
        response.raise_for_status()

        # JSON 응답 파싱
        result = response.json()

        # 응답 데이터에서 content 추출
        if result and 'choices' in result and result['choices']:
            # LangChain ChatMessage 객체 대신, content 문자열만 반환
            return result['choices'][0]['message']['content']
        else:
            return "Error: Could not parse model response."

    except requests.exceptions.HTTPError as err:
        print(f"HTTP Error occurred: {err}")
        print(f"Response Content: {response.text[:200]}...")  # 에러 내용 일부 출력
    except Exception as err:
        print(f"An unexpected error occurred: {err}")

    return "An error occurred during API call."

In [5]:
llm = solar_invoke

print("--- SOLAR Invoke 테스트 ---")
prompt_text = "안녕하세요. 저는 파이썬 개발자입니다. 저에게 맞는 재미있는 기술 질문 하나를 해주세요."
final_answer = solar_invoke(prompt_text)

print("\n[최종 SOLAR 응답]:")
print(final_answer)

--- SOLAR Invoke 테스트 ---

[최종 SOLAR 응답]:
안녕하세요! 파이썬 개발자님을 위해 **"재미있는 동시성 퍼즐"**을 준비했어요.  
다음 코드를 분석하고 문제점과 해결 방법을 설명해보세요:

```python
import threading
import time

counter = 0

def increment():
    global counter
    for _ in range(100000):
        counter += 1

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)

t1.start()
t2.start()
t1.join()
t2.join()

print("Final counter:", counter)
```

**예상 출력**은 `200000`이지만, 실제로는 **덜 나오는 경우가 많습니다**. 왜 이런 현상이 발생하며, 어떻게 수정해야 할까요?  
(힌트: GIL과 원자성(atomicity)을 고려해보세요)

---

### 🔍 추가 도전 과제
1. `threading.Lock` 대신 **`concurrent.futures`**로 해결해보세요.
2. 멀티프로세스 환경에서 동일한 코드를 작성한다면 어떻게 달라질까요?  
3. 만약 `counter += 1`이 아닌 `counter += some_shared_list.pop()`이라면 어떤 문제가 더 발생할까요?

즐겁고 유익한 고민 되세요! 😊
